In [ ]:
from diffusers.utils import load_image, make_image_grid
from diffusers import StableDiffusionInstructPix2PixPipeline

import mimetypes
import torch
from pathlib import Path

from typing import Union

from IPython.display import display

In [ ]:


def collect_images(path: Union[str, Path]) -> list[Path]:
    mime_checker = mimetypes.MimeTypes()

    def validate_file_type(path: Path):
        mime_type = mime_checker.guess_type(path)[0]
        return mime_type is not None and mime_type.startswith("image")

    path = Path(path)
    if path.is_dir():
        return sorted(filter(validate_file_type, path.rglob("*.*")))
    else:
        return [path]

In [ ]:
root_path = Path("~/data/Weather/Weather_Anti_UAV/backgrounds/FWID/rainy").expanduser()
image_paths = collect_images(root_path)[:5]

images = [load_image(image_path.as_posix()) for image_path in image_paths]
for image in images:
    print(image.size)
make_image_grid(images, rows=1, cols=len(images), resize=512)

In [ ]:
pipeline = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    "timbrooks/instruct-pix2pix", torch_dtype=torch.float16, use_safetensors=True
)
pipeline.enable_model_cpu_offload(device="cuda:4")

In [ ]:
generator = torch.Generator().manual_seed(1)
prompt = "Remove the rain"
#negative_prompt = "Keep the structure"

for image in images:
    base_image = pipeline(prompt, image=image, generator=generator, image_guidance_scale=1.5).images[0]
    display(make_image_grid([image, base_image], rows=1, cols=2))